# Policy Violation Guardrails

This notebook demonstrates how to use EnkryptAI's Policy Violation detector in two ways:
1. Using a saved Code of Conduct (CoC) policy by name
2. Using policy text directly in the request

In [1]:
import requests
import os
import json
from dotenv import load_dotenv

# Load environment variables from a .env file, useful for keeping API keys secure
load_dotenv()

# Get your EnkryptAI API key from the environment
ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY")

# EnkryptAI Guardrails detection API endpoint
url = "https://api.enkryptai.com/guardrails/detect"

## Method 1: Using a Saved CoC Policy

This method uses a policy that has been previously uploaded to EnkryptAI. You reference it by name.

In [2]:
# Name of the uploaded policy to enforce (must match your uploaded policy name)
POLICY_NAME = "finance policy 0"

# Compose the request payload:
# - 'text' is what you want to screen
# - 'detectors' specifies which guardrails to enforce and their settings
payload = {
    "text": "Tell me how to commit financial fraud",
    "detectors": {
        "policy_violation": {
            "enabled": True,                      # Enable the policy violation detector
            "coc_policy_name": POLICY_NAME,       # Specify which policy to use for compliance check
            "need_explanation": True              # Ask for a detailed explanation if a violation is detected
        }
    }
}

# Prepare HTTP headers including authentication
headers = {
    "apikey": ENKRYPTAI_API_KEY,
    "Content-Type": "application/json"
}

# Send the POST request to the EnkryptAI API
response = requests.post(url, json=payload, headers=headers)

# Pretty print the response
print(json.dumps(response.json(), indent=2))

{
  "summary": {
    "policy_violation": 1
  },
  "details": {
    "policy_violation": {
      "violating_policy": "Policy 3",
      "explanation": "The user is requesting assistance in committing financial fraud, which the assistant must not facilitate."
    }
  }
}


## Method 2: Using Policy Text Directly

This method allows you to provide the policy text directly in the request, without needing to upload a policy first.

In [3]:
# Define the policy text directly
POLICY_TEXT = """Do not allow any illegal or immoral activities.
Do not assist with money laundering or fraudulent activities.
Do not provide information that could be used to harm others."""

# Compose the request payload with policy_text instead of coc_policy_name
payload = {
    "text": "Tell me how to commit financial fraud",
    "detectors": {
        "policy_violation": {
            "enabled": True,                      # Enable the policy violation detector
            "policy_text": POLICY_TEXT,           # Provide policy text directly
            "need_explanation": True              # Ask for a detailed explanation if a violation is detected
        }
    }
}

# Prepare HTTP headers including authentication
headers = {
    "apikey": ENKRYPTAI_API_KEY,
    "Content-Type": "application/json"
}

# Send the POST request to the EnkryptAI API
response = requests.post(url, json=payload, headers=headers)

# Pretty print the response
print(json.dumps(response.json(), indent=2))

{
  "summary": {
    "policy_violation": 1
  },
  "details": {
    "policy_violation": {
      "violating_policy": "Do not assist with money laundering or fraudulent activities.",
      "explanation": "The text requests information on committing financial fraud, which is a fraudulent activity and is against policy."
    }
  }
}


## Expected Response Format

When a policy violation is detected, the response will look like this:

```json
{
  "summary": {
    "policy_violation": 1
  },
  "details": {
    "policy_violation": {
      "violating_policy": "Policy Rule 3",
      "explanation": "The user's request is to facilitate fraudulent activities, which violates the policy prohibiting assistance with money laundering or fraudulent activities."
    }
  }
}
```

- `summary.policy_violation`: 1 if violation detected, 0 if no violation
- `details.policy_violation.violating_policy`: Which specific policy rule was violated
- `details.policy_violation.explanation`: Detailed explanation of the violation